# 第6课 · 最美公式为你转动——欧拉公式（Euler's formula）e^{iθ}=cosθ+isinθ 与 FFT 的旋转因子（twiddle factor）

**学习目标**
1. 理解欧拉公式 `e^{iθ} = cosθ + i·sinθ`——复指数是单位圆上的旋转，实部=cosθ，虚部=sinθ
2. 手动验证 θ=0, π/2, π, 3π/2 等特殊角度，确认实部、虚部与三角函数逐行吻合
3. 实现 `twiddle(k, n, N)` 旋转因子 `e^{-2πikn/N}`，掌握 DFT 中频率索引 k、时间索引 n、长度 N 的含义
4. 确认旋转因子模长恒为 1，相位均匀递减，步长为 `-2π/N`
5. 建立"旋转因子 → DFT 矩阵 → FFT"的直觉链，为 L37–L39 手写 FFT 做准备

**为什么对 Aurora 重要**：DFT 的核心就是旋转因子 `e^{−2πi·k·n/N}`，FFT 全靠它。理解它，L37-L39 重写 FFT 才不虚。

← **上一课**　[L05 · 复数的模与相位](L05_complex_numbers.ipynb)

> 上节课学习了 **复数的模与相位**：给每个频率发身份证，从复数读出模与相位——FFT 的母语。  
> 本课将探讨 **欧拉公式 e^{iθ}=cosθ+isinθ**。

## 本课剧情：一行数字，打包了圆上的一切

你有没有想过，为什么 Euler 公式 e^{iθ} 被称为"数学最美公式"？

理由很朴素：**一个实数 θ，就能确定圆周上的完整位置。**

单位圆上角度 θ 处的点，坐标是两个数：(cos θ, sin θ)。普通写法需要两个格子。  
欧拉的天才之处：用一个复数把这两个格子合并：

```
e^{iθ} = cos θ + i·sin θ
```

这不只是符号简写——它揭示了旋转的代数结构：
- **乘法 = 旋转**：`e^{iα} × e^{iβ} = e^{i(α+β)}`（角度相加 = 两次旋转合并）
- **n 次幂 = n 倍旋转**：`(e^{iθ})^n = e^{inθ}`

DFT 的核心运算就是把信号与一列"旋转因子"相乘，逐个投影到各个频率方向上。旋转因子（twiddle factor）就是 `W = e^{−2πi·k·n/N}`。理解了 e^{iθ}，DFT 就不再神秘。

## 1. 手动拼 e^{iθ}，验证 = np.exp(iθ)

先不用任何公式，从几何出发：

角度 θ 处的单位圆上的点坐标 = (cos θ, sin θ)。

现在"打包"这两个数成一个复数：`manual = cos θ + i·sin θ`

Euler 公式断言：`np.exp(1j * theta) == manual`

验证很简单——运行下面的格，确认两列数字完全相同。这不是凑巧，是数学定理。

> **为什么 e^{iθ} 等于 cos+i·sin？**  
> 泰勒展开 e^x = 1 + x + x²/2! + ...，代入 x = iθ，实部收敛到 cos 级数，虚部收敛到 sin 级数。  
> L06 不推导这个——只用它、验证它；L37 推导 DFT 时会直接调用这个结论。

## 实验入口：角度、坐标和旋转

同一组角度同时打印为 `cos(θ)`、`sin(θ)` 和 `np.exp(1j*θ)`，逐行对齐三列，观察实部与 cos 完全相等、虚部与 sin 完全相等。

In [ ]:
import numpy as np
theta = np.linspace(0, 2*np.pi, 9)
manual = np.cos(theta) + 1j*np.sin(theta)
print('与 np.exp(iθ) 一致:', np.allclose(manual, np.exp(1j*theta)))
print('每个点模长都是1:', np.allclose(np.abs(manual), 1.0))

## 动手观察：复数就是"旋转的位置"

单位圆上角度 θ 的点坐标是 (cosθ, sinθ)，欧拉公式把这两个实数合并写作 e^(iθ)。下面验证几个特殊角度：0、π/2、π、3π/2。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：角度、三角函数和复数坐标对齐

打印同一组角度的 `cos(θ)`、`sin(θ)` 和 `np.exp(1j*θ)` 三列，核对实部、虚部与三角函数的数值是否逐行吻合。

In [ ]:
import numpy as np

angles = np.linspace(0, 2*np.pi, 9)
for theta in angles:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} | cos={z.real:6.3f} | sin={z.imag:6.3f} | z={z.real:6.3f}{z.imag:+6.3f}j')


## 2. ✏️ 实现 FFT 旋转因子 `twiddle(k, n, N)`

`W = e^{−2πi·k·n/N}`

**推理路线**：
1. 计算相位（phase，φ）角：`θ = -2π · k · n / N`（k 是频率索引（frequency index，k），n 是采样点索引，N 是 DFT 总点数）
2. 对相位角取复指数：`np.exp(1j * θ)` = `cosθ + i·sinθ`，结果模（magnitude，r）长恒为 1，落在单位圆上
3. 注意负号由 DFT 定义决定，不能省；`k · n / N` 控制本次旋转在整圈中的比例

**参考输入输出**：`twiddle(k=1, n=1, N=8)` = `exp(-2πi/8)` ≈ `0.707 - 0.707j`（位于单位圆第四象限 45° 处）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `twiddle` 前明确三件事：
- 输入：`k`（频率索引）、`n`（时间样本索引）、`N`（DFT 总长度）
- 关键步骤：计算相位 `-2π·k·n/N`，对其取复指数 `np.exp(...)`
- 返回：一个复数，模为 1，辐角为 `-2πkn/N`

In [ ]:
def twiddle(k, n, N):
    # ✏️ TODO: 返回 e^{-2πi·k·n/N}
    raise NotImplementedError("TODO: 返回 e^{-2πi·k·n/N} — 旋转因子复数值")


In [ ]:
import numpy as np
try:
    assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'k·n=0 应为 1'
    assert abs(abs(twiddle(3, 5, 8)) - 1) < 1e-12, '模长应为 1'
    _ref = np.exp(-2j * np.pi * 1 * 1 / 8)
    assert abs(twiddle(1, 1, 8) - _ref) < 1e-12, '相位方向错误：公式是 e^{-2πikn/N}，负号不能省'
    print('✅ 通过：你握住了 FFT 的旋转因子。')
except NotImplementedError:
    print('⬜ 请先实现 twiddle(k, n, N)，再运行验证格')


## 3. 画出单位圆

In [ ]:
import numpy as np, matplotlib.pyplot as plt
fine = np.exp(1j*np.linspace(0, 2*np.pi, 200))
plt.figure(figsize=(4,4))
plt.plot(fine.real, fine.imag); plt.gca().set_aspect('equal')
plt.grid(True, alpha=.3); plt.title('e^{iθ} on the unit circle'); plt.show()

**🔗 Aurora 连接**：朴素 DFT（`src/aurora/audio/transforms.py` 中的 `dft()`）对每个频率索引 k，将信号与一列旋转因子 `twiddle(k, 0..N-1, N)` 做内积——内部用 `np.outer(k_vec, n_vec)` 构造完整旋转因子矩阵，与本课的 `twiddle(k, n, N)` 在数学上完全等价。L37-L39 你会把这个 O(N²) 矩阵乘法重写成 O(N log N) 的 FFT——旋转因子的定义不变，变的是计算顺序。

In [ ]:
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z = np.exp(1j*theta)
    reconstructed = np.cos(theta) + 1j*np.sin(theta)
    print(f'theta={theta:.2f} | exp={z:.2f} | cos+i sin={reconstructed:.2f} | match={np.allclose(z, reconstructed)}')


## 参数实验：观察相位均匀递减

固定 `k=1`，让 `n` 从 0 到 N-1 遍历，打印每个旋转因子的相位（`np.angle`，单位：弧度（radian）），确认相位均匀递减，步长为 `-2π/N`：

```python
N = 8
for n in range(N):
    w = twiddle(k=1, n=n, N=N)
    print(f'n={n} | angle={np.angle(w):.4f} rad ({np.degrees(np.angle(w)):.1f}°)')
# 预期：每行角度差 -45°（= -2π/8 ≈ -0.785 rad）
```

再把 N 改成 4（步长变 -90°）或 1024（步长缩到约 -0.35°），感受 N 对频率分辨率的控制。

In [ ]:
import numpy as np

try:
    # 学习目标 #4：确认旋转因子相位均匀递减，步长为 -2π/N
    N = 8
    for n in range(N):
        w = twiddle(k=1, n=n, N=N)
        print(f'n={n} | angle={np.angle(w):.4f} rad ({np.degrees(np.angle(w)):.1f}°)')

    phases = [np.angle(twiddle(1, n, 8)) for n in range(8)]
    diffs = np.diff(np.unwrap(phases))
    assert np.allclose(diffs, -2 * np.pi / 8), '相位步长不均匀，检查 twiddle 实现'
    print(f'\n✅ 相位步长均匀，每步 {-2*np.pi/8:.4f} rad（= -2π/8）')
    print('再把 N 改成 4 或 1024，感受 N 对频率分辨率的控制。')
except NotImplementedError:
    print('⬜ 请先实现 twiddle(k, n, N)，再运行相位实验')


## 本课收束

现在可以用 `np.exp(1j*theta)` 从任意角度算出单位圆上的复数坐标，也可以用 `twiddle(k, n, N)` 算出 DFT 的旋转因子 e^(-2πikn/N)。`twiddle` 直接对应 Aurora 音频核中 `transforms.py` 的 `dft()` 实现逻辑，L37-L39 的 FFT 实现会把它复用。下一课：**L07** 方波谐波叠加——用正弦波叠加近似方波，直观演示傅里叶级数的合成过程，建立「任何周期信号都是正弦波之和」的直觉。

`twiddle` 直接对应 `L37_dft.ipynb` 中 DFT 矩阵的构造——每一行都是一组旋转因子，与 `twiddle(k, n, N)` 数学等价。

## ✏️ 白板挑战：旋转因子手算（目标 8 分钟）

盖上屏幕，纸上作答：

**参数**：N = 8（8 点 DFT），k = 1，n = 0, 1, 2, 3

**问 1**：写出旋转因子公式 `W(k,n,N) = e^{-2πi·k·n/N}`，代入 k=1, N=8，化简指数部分。

**问 2**：计算 W(1, 0, 8)、W(1, 1, 8)、W(1, 2, 8)、W(1, 3, 8) 的值（极坐标或直角坐标均可）。
提示：先手算，再运行下方对答案格；需要公式提示时展开下方折叠块。

<details><summary>卡住时可展开（先尽量不用）</summary>

W(1,n,8) = e^{-iπn/4}；n=0→1，n=1→e^{-iπ/4}=cos(-45°)+i·sin(-45°)…

</details>

**问 3**：这 8 个旋转因子（n=0…7）均匀分布在单位圆的哪个方向？顺时针还是逆时针？

**问 4**：若 k=0，所有旋转因子 W(0,n,N) 等于多少？为什么？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

N = 8

# 问1+问2：旋转因子手算验证
print("Q1/Q2 ✅  W(1,n,8) = e^{-iπn/4}：")
for n in range(4):
    w_theory = np.exp(-1j * 2 * np.pi * 1 * n / N)
    w_code   = twiddle(1, n, N)
    assert np.isclose(w_code, w_theory, atol=1e-10), f"n={n}: 不一致"
    deg = np.degrees(np.angle(w_theory))
    print(f"  n={n}: e^{{-iπ·{n}/4}} = {w_theory.real:+.4f}{w_theory.imag:+.4f}j  (相位 {deg:+.1f}°)")

# 问3：8个旋转因子均匀分布（先 unwrap 解绕，避免 ±π 跳变污染步长）
phases = np.unwrap([np.angle(twiddle(1, n, N)) for n in range(N)])
diffs = np.diff(phases)
step_deg = np.degrees(diffs[0])
all_equal = np.allclose(diffs, diffs[0], atol=1e-10)
assert all_equal, "相位步长不均匀"
print(f"\nQ3 ✅  8个旋转因子均匀分布，步长 {step_deg:.1f}°（{'顺时针' if step_deg < 0 else '逆时针'}）")

# 问4：k=0时所有因子为1
for n in range(N):
    w0 = twiddle(0, n, N)
    assert np.isclose(w0, 1.0, atol=1e-12), f"k=0,n={n}: 期望1，得到{w0}"
print(f"\nQ4 ✅  k=0 时 W(0,n,N) = e^0 = 1（所有时刻都不旋转 → DC分量）")
print("\n🎉 旋转因子白板挑战通过！DFT 的核心运算已在脑中具象化。")

In [ ]:
# ✏️ 本课自评
l06_review = {
    "euler_formula_verified":   None,  # 验证了 e^{iθ} = cosθ+i·sinθ？True/False
    "twiddle_implemented":      None,  # twiddle(k,n,N) 实现并通过断言？True/False
    "rotation_intuition":       None,  # 理解"乘复数=旋转"？True/False
    "whiteboard_passed":        None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l06_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l06_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L06 全部通关！进入 L07：万物皆正弦')

---

→ **下一课**　[L07 · 万物皆正弦](L07_fourier_intuition.ipynb)

> 下节课将学习 **万物皆正弦**：用正弦波谐波叠加合成方波，傅里叶直觉一图彻底建立。